In [ ]:
# Environment Detection & Setup
import sys
import os

# Detect if running in Colab or local
try:
 import google.colab
 IN_COLAB = True
 print(" Running in Google Colab")
except:
 IN_COLAB = False
 print(" Running in Local Environment")

# Mount Google Drive if in Colab
if IN_COLAB:
 from google.colab import drive
 drive.mount('/content/drive')

 # Verify GPU
 import tensorflow as tf
 gpus = tf.config.list_physical_devices('GPU')
 if gpus:
 print(f" GPU Detected: {gpus[0].name}")
 else:
 print(" No GPU detected - using CPU")
else:
 print(" Local environment - configure paths accordingly")

In [ ]:
# Essential Imports
import os
import numpy as np
import pandas as pd!pip install pydicom -q
import pydicom
from pathlib import Path
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# Deep learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import BinaryAccuracy, Precision, Recall, AUC

# Metrics & visualization
from sklearn.metrics import roc_auc_score, roc_curve, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns

# Set seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

In [ ]:
# V4 CONFIGURATION - LITERATURE-BACKED
from pathlib import Path

class Config:
 """Configuration for V4 LITERATURE-BACKED mammography classification
 
 LITERATURE REFERENCES:
 - McKinney et al., Nature 2020 (Google Health)
 - Swapna Rani et al., JATIT 2025 (ILB-BCD, 99.16% accuracy)
 - Wu et al., IEEE TMI 2019 (AUC 0.895)
 
 KEY V4 CHANGES FROM V3:
 1. REMOVED class_weight from model.fit() - use ONLY focal loss
 2. Stage 2b LR: 1e-5 → 1e-6 (prevents catastrophic forgetting)
 3. Keep first 200 layers FROZEN during Stage 2b
 4. L1/L2 regularization: 0.01 (per ILB-BCD paper)
 5. Gradient clipping: clipnorm=1.0
 6. Resume from Stage 2a checkpoint
 """

 def __init__(self):
 # Detect environment
 try:
 import google.colab
 self.IN_COLAB = True
 except:
 self.IN_COLAB = False

 # Set paths based on environment
 if self.IN_COLAB:
 self.BASE_PATH = Path('/content/drive/MyDrive/CBIS-DDSM-data')
 self.DICOM_PATH = self.BASE_PATH / 'CBIS-DDSM'
 self.CSV_PATH = self.BASE_PATH / 'csv'
 self.EDA_OUTPUT_PATH = self.BASE_PATH / 'eda_complete'
 self.OUTPUT_PATH = self.BASE_PATH / 'model_outputs_v4'
 self.CHECKPOINT_PATH = self.BASE_PATH / 'checkpoints_v4'
 self.RESULTS_PATH = self.BASE_PATH / 'results_v4'
 # V4: Path to V3 checkpoints for resuming
 self.V3_CHECKPOINT_PATH = self.BASE_PATH / 'checkpoints'
 else:
 self.BASE_PATH = Path('/home/tars/Desktop/final_project/CBIS-DDSM model training')
 self.DICOM_PATH = Path('~/GoogleDrive/CBIS-DDSM-data/CBIS-DDSM').expanduser()
 self.CSV_PATH = Path('~/GoogleDrive/CBIS-DDSM-data/csv').expanduser()
 self.EDA_OUTPUT_PATH = Path('~/GoogleDrive/CBIS-DDSM-data/eda_complete').expanduser()
 self.OUTPUT_PATH = self.BASE_PATH / 'local_outputs_v4'
 self.CHECKPOINT_PATH = self.BASE_PATH / 'local_checkpoints_v4'
 self.RESULTS_PATH = self.BASE_PATH / 'local_results_v4'
 self.V3_CHECKPOINT_PATH = self.BASE_PATH / 'checkpoints_v2'

 # Data files
 self.CALC_TRAIN_CSV = self.CSV_PATH / 'calc_case_description_train_set.csv'
 self.CALC_TEST_CSV = self.CSV_PATH / 'calc_case_description_test_set.csv'
 self.MASS_TRAIN_CSV = self.CSV_PATH / 'mass_case_description_train_set.csv'
 self.MASS_TEST_CSV = self.CSV_PATH / 'mass_case_description_test_set.csv'
 self.METADATA_CSV = self.CSV_PATH / 'metadata.csv'

 # Image parameters (Literature: 224x224 is standard)
 self.IMG_SIZE = (224, 224)
 self.IMG_CHANNELS = 3

 # V4 TRAINING STRATEGY: 3-STAGE WITH LITERATURE FIXES
 self.BATCH_SIZE = 32 # Literature: 32 is standard
 self.STAGE1_EPOCHS = 30
 self.STAGE2A_EPOCHS = 30
 self.STAGE2B_EPOCHS = 50 # Can extend to 100 per ILB-BCD
 
 # V4 FIX: Learning rates (per literature)
 self.STAGE1_LR = 1e-3 # Higher LR for randomly initialized head
 self.STAGE2A_LR = 1e-4 # ILB-BCD paper: 0.0001
 self.STAGE2B_LR = 1e-6 # V4 FIX: 10x lower than V3 (was 1e-5)
 
 # Data split (Literature: 70-15-15)
 self.TRAIN_SPLIT = 0.70
 self.VAL_SPLIT = 0.15
 self.TEST_SPLIT = 0.15

 # V4 FIX: REGULARIZATION (ILB-BCD paper: L1=L2=0.01)
 self.L1_REGULARIZATION = 0.01 # V4: Increased from 1e-5
 self.L2_REGULARIZATION = 0.01 # V4: Increased from 1e-4
 self.DROPOUT_RATE = 0.4
 
 # V4 FIX: Gradient clipping to prevent exploding gradients
 self.GRADIENT_CLIPNORM = 1.0
 
 # Label smoothing (helps calibration)
 self.LABEL_SMOOTHING = 0.1

 # Augmentation (conservative for medical imaging)
 self.ROTATION_RANGE = 15
 self.ZOOM_RANGE = [0.9, 1.1]
 self.BRIGHTNESS_RANGE = [0.9, 1.1]
 self.HORIZONTAL_FLIP = True
 self.USE_DENSENET_PREPROCESSING = True

 # V4 FIX: CLASS WEIGHTING STRATEGY
 # CRITICAL: Use ONLY focal loss alpha, NOT class_weight in fit()
 # V3 BUG: Used BOTH → ~10x effective weight → catastrophic forgetting
 self.USE_CLASS_WEIGHT_IN_FIT = False # V4: DISABLED!
 self.MALIGNANT_WEIGHT_MULTIPLIER = 3.0 # For reference only
 
 # Focal Loss: alpha handles class imbalance, gamma focuses on hard examples
 self.FOCAL_LOSS_GAMMA = 2.0
 self.FOCAL_LOSS_ALPHA = 0.75 # 75% weight on malignant class

 # V4 FIX: IMPROVED UNFREEZING STRATEGY
 # Keep early layers FROZEN even in Stage 2b
 # These contain generic features (edges, textures) that transfer well
 self.FREEZE_FIRST_N_LAYERS = 200 # Keep conv1, dense_block1-2 frozen

 # Callbacks
 self.EARLY_STOP_PATIENCE = 20
 self.LR_REDUCE_PATIENCE = 7
 self.LR_REDUCE_FACTOR = 0.5
 self.MIN_EPOCHS_STAGE1 = 20
 self.MIN_EPOCHS_STAGE2A = 20
 self.MIN_EPOCHS_STAGE2B = 30

 # Resume capability
 self.RESUME_FROM_STAGE = None # Set to 'stage2a' to resume

 # Create output directories
 self._create_dirs()

 def _create_dirs(self):
 """Create output directories if they don't exist"""
 for path in [self.OUTPUT_PATH, self.CHECKPOINT_PATH, self.RESULTS_PATH]:
 path.mkdir(parents=True, exist_ok=True)

 def print_config(self):
 """Print current configuration"""
 print("\n" + "="*70)
 print(" V4 LITERATURE-BACKED MAMMOGRAPHY CLASSIFICATION")
 print("="*70)
 print(f"\n Based on:")
 print(f" • McKinney et al., Nature 2020 (Google Health)")
 print(f" • Swapna Rani et al., JATIT 2025 (ILB-BCD, 99.16%)")
 print(f" • Wu et al., IEEE TMI 2019 (AUC 0.895)")
 print(f"\n Paths:")
 print(f" Environment: {'Google Colab' if self.IN_COLAB else 'Local'}")
 print(f" Checkpoint Path: {self.CHECKPOINT_PATH}")
 print(f"\n V4 TRAINING STRATEGY:")
 print(f" Stage 1: {self.STAGE1_EPOCHS} epochs, LR={self.STAGE1_LR} (head only)")
 print(f" Stage 2a: {self.STAGE2A_EPOCHS} epochs, LR={self.STAGE2A_LR} (last block)")
 print(f" Stage 2b: {self.STAGE2B_EPOCHS} epochs, LR={self.STAGE2B_LR} (partial unfreeze)")
 print(f"\n V4 KEY FIXES:")
 print(f" class_weight in fit(): DISABLED (was causing double weighting)")
 print(f" Stage 2b LR: {self.STAGE2B_LR} (10x lower than V3)")
 print(f" Freeze first {self.FREEZE_FIRST_N_LAYERS} layers in Stage 2b")
 print(f" L1/L2 regularization: {self.L1_REGULARIZATION} (per ILB-BCD paper)")
 print(f" Gradient clipping: clipnorm={self.GRADIENT_CLIPNORM}")
 print(f"\n CLASS WEIGHTING (Focal Loss Only):")
 print(f" Focal Loss α: {self.FOCAL_LOSS_ALPHA} (75% weight on malignant)")
 print(f" Focal Loss γ: {self.FOCAL_LOSS_GAMMA} (focus on hard examples)")
 print(f"\n TARGET METRICS:")
 print(f" Sensitivity: 96.2% | Specificity: 82.5%")
 print(f" NPV: 99.1% | AUROC: 0.93")
 print("="*70 + "\n")

# Initialize configuration
config = Config()
config.print_config()

# Set GPU memory growth
gpus = tf.config.list_physical_devices('GPU')
if gpus:
 for gpu in gpus:
 tf.config.experimental.set_memory_growth(gpu, True)
 print(f" GPU configured with memory growth")

## Data Loading Functions

In [ ]:
# Data Loading Utilities (V3 Compatible)
# Uses the same ROI finding approach as V3 - based on case_dir folder names

import cv2
from pathlib import Path
from tensorflow.keras.applications.densenet import preprocess_input as densenet_preprocess

def find_dicom_file(case_dir_name, base_path):
 """
 Find the ROI DICOM file in case directory with 3-level nesting.
 
 CRITICAL FIX: Load ROI crop, NOT full mammogram!
 
 In CBIS-DDSM, ROI folders contain:
 - 1-1.dcm: Full mammogram (~4800x3000 pixels) - DON'T USE THIS!
 - 1-2.dcm: ROI crop (~500x400 pixels) - USE THIS!
 
 Structure: case_dir / date / series / file.dcm
 
 Args:
 case_dir_name: Name of the case directory (e.g., "Mass-Training_P_00001_LEFT_CC_1")
 base_path: Base path to DICOM dataset
 
 Returns:
 Path to ROI DICOM file, or None if not found
 """
 case_path = Path(base_path) / case_dir_name
 
 if not case_path.exists():
 return None
 
 try:
 # Collect ALL DICOM files in this case directory
 dcm_files = list(case_path.rglob("*.dcm"))
 
 if not dcm_files:
 return None
 
 # Sort files by name to ensure consistent ordering
 # In CBIS-DDSM: 1-1.dcm = full mammogram, 1-2.dcm = ROI crop
 dcm_files = sorted(dcm_files, key=lambda x: x.name)
 
 # If multiple DICOM files exist, return the LAST one (ROI crop)
 if len(dcm_files) >= 2:
 return dcm_files[-1] # Return 1-2.dcm (ROI crop)
 else:
 return dcm_files[0] # Single file case
 
 except Exception as e:
 print(f"Error accessing {case_dir_name}: {e}")
 return None


def apply_dicom_corrections(dcm, img):
 """Apply DICOM-specific corrections to raw pixel data."""
 # Apply Rescale Slope and Intercept
 rescale_slope = float(getattr(dcm, 'RescaleSlope', 1.0))
 rescale_intercept = float(getattr(dcm, 'RescaleIntercept', 0.0))
 
 if rescale_slope!= 1.0 or rescale_intercept!= 0.0:
 img = img * rescale_slope + rescale_intercept
 
 # Apply VOI LUT or Window/Level
 voi_lut_applied = False
 
 if hasattr(dcm, 'VOILUTSequence') and dcm.VOILUTSequence:
 try:
 from pydicom.pixel_data_handlers.util import apply_voi_lut
 img = apply_voi_lut(img.astype(np.float64), dcm, index=0).astype(np.float32)
 voi_lut_applied = True
 except:
 pass
 
 if not voi_lut_applied:
 window_center = getattr(dcm, 'WindowCenter', None)
 window_width = getattr(dcm, 'WindowWidth', None)
 
 if window_center is not None and window_width is not None:
 if hasattr(window_center, '__iter__') and not isinstance(window_center, str):
 window_center = float(window_center[0])
 else:
 window_center = float(window_center)
 
 if hasattr(window_width, '__iter__') and not isinstance(window_width, str):
 window_width = float(window_width[0])
 else:
 window_width = float(window_width)
 
 if window_width > 0:
 img_min = window_center - window_width / 2
 img_max = window_center + window_width / 2
 img = np.clip(img, img_min, img_max)
 img = (img - img_min) / (img_max - img_min)
 voi_lut_applied = True
 
 # Handle Photometric Interpretation
 if hasattr(dcm, 'PhotometricInterpretation'):
 if dcm.PhotometricInterpretation == "MONOCHROME1":
 if voi_lut_applied:
 img = 1.0 - img
 else:
 img = np.max(img) - img
 
 return img, voi_lut_applied


def load_dicom_image(case_dir_name, base_path, target_size=(224, 224)):
 """
 Load and preprocess DICOM image following research paper methodology.
 """
 try:
 dcm_path = find_dicom_file(case_dir_name, base_path)
 
 if dcm_path is None:
 return None
 
 dcm = pydicom.dcmread(str(dcm_path))
 img = dcm.pixel_array.astype(np.float32)
 
 if img.size == 0:
 return None
 
 # Apply DICOM corrections
 img, voi_applied = apply_dicom_corrections(dcm, img)
 
 # Normalize if VOI LUT wasn't applied
 if not voi_applied:
 img_p1 = np.percentile(img, 1)
 img_p99 = np.percentile(img, 99)
 
 if img_p99 > img_p1:
 img = (img - img_p1) / (img_p99 - img_p1)
 else:
 img_min, img_max = img.min(), img.max()
 if img_max > img_min:
 img = (img - img_min) / (img_max - img_min)
 else:
 img = np.full_like(img, 0.5)
 
 img = np.clip(img, 0.0, 1.0)
 
 # Resize to target size
 img = cv2.resize(img, target_size, interpolation=cv2.INTER_LANCZOS4)
 
 # Convert to RGB
 img_rgb = np.stack([img, img, img], axis=-1)
 
 # Apply DenseNet preprocessing
 img_rgb = img_rgb * 255.0
 img_rgb = densenet_preprocess(img_rgb)
 
 return img_rgb
 
 except Exception as e:
 return None


print(" Data loading utilities defined (V3 compatible)")

In [ ]:
# Load and Prepare Dataset (V3 Compatible Approach)
# Uses case_dir folder names like V3, NOT raw CSV paths

def extract_case_dir_from_path(path_str):
 """
 Extract case directory name from CSV path.
 
 CSV paths look like:
 'Calc-Training_P_00005_LEFT_MLO_1/1.3.6.../1-1.dcm'
 
 Extracted: 'Calc-Training_P_00005_LEFT_MLO_1' (the case folder name)
 """
 if pd.isna(path_str) or path_str == '':
 return None
 
 # Split by '/' and get the first part (case directory name)
 parts = str(path_str).split('/')
 if len(parts) >= 1:
 return parts[0]
 return None


def load_cbis_ddsm_data(config):
 """
 Load CBIS-DDSM dataset using V3-compatible approach.
 
 Key fix: Extract case_dir folder name from CSV path, then use
 find_dicom_file() which searches within that folder for ROI crops.
 """
 
 print("\n" + "="*70)
 print(" LOADING CBIS-DDSM DATASET (V3 Compatible)")
 print("="*70)
 
 # Load CSV files
 print("\n Loading CSV metadata...")
 
 dfs = []
 csv_files = [
 (config.CALC_TRAIN_CSV, 'calc', 'train'),
 (config.CALC_TEST_CSV, 'calc', 'test'),
 (config.MASS_TRAIN_CSV, 'mass', 'train'),
 (config.MASS_TEST_CSV, 'mass', 'test'),
 ]
 
 for csv_path, lesion_type, split in csv_files:
 if csv_path.exists():
 df = pd.read_csv(csv_path)
 df['lesion_type'] = lesion_type
 df['original_split'] = split
 dfs.append(df)
 print(f" Loaded {csv_path.name}: {len(df)} records")
 else:
 print(f" Missing {csv_path.name}")
 
 if not dfs:
 raise FileNotFoundError("No CSV files found!")
 
 # Combine all data
 combined_df = pd.concat(dfs, ignore_index=True)
 print(f"\n Total records: {len(combined_df)}")
 
 # Extract pathology (BENIGN/MALIGNANT)
 combined_df['label'] = combined_df['pathology'].apply(
 lambda x: 1 if 'MALIGNANT' in str(x).upper() else 0
 )
 
 # Find the column with cropped image paths
 path_col = None
 for col in ['cropped image file path', 'ROI mask file path', 'image file path']:
 if col in combined_df.columns:
 path_col = col
 print(f"\n Using path column: '{col}'")
 break
 
 if path_col is None:
 raise ValueError("Could not find image path column in CSV!")
 
 # Extract case_dir from paths (like V3 does)
 print(f" Extracting case directories from '{path_col}'...")
 combined_df['case_dir'] = combined_df[path_col].apply(extract_case_dir_from_path)
 
 # Remove rows without valid case_dir
 valid_mask = combined_df['case_dir'].notna()
 combined_df = combined_df[valid_mask].copy()
 print(f" Records with valid case_dir: {len(combined_df)}")
 
 # Find ROI DICOM files using V3 approach
 print(f"\n Finding ROI DICOM files in: {config.DICOM_PATH}")
 
 valid_samples = []
 missing_count = 0
 
 for idx, row in combined_df.iterrows():
 case_dir = row['case_dir']
 
 if pd.isna(case_dir):
 continue
 
 # Use V3's find_dicom_file approach
 dcm_path = find_dicom_file(case_dir, config.DICOM_PATH)
 
 if dcm_path:
 valid_samples.append({
 'case_dir': case_dir,
 'path': str(dcm_path),
 'label': row['label'],
 'lesion_type': row['lesion_type'],
 'pathology': row['pathology'],
 'original_split': row['original_split']
 })
 else:
 missing_count += 1
 if missing_count <= 5: # Only show first 5 missing
 print(f" Not found: {case_dir}")
 
 if missing_count > 5:
 print(f"... and {missing_count - 5} more missing")
 
 print(f"\n Found {len(valid_samples)} valid ROI samples")
 print(f" Missing {missing_count} samples")
 
 if len(valid_samples) == 0:
 # Debug: show what directories exist
 print(f"\n DEBUG: Checking DICOM path structure...")
 dicom_path = Path(config.DICOM_PATH)
 if dicom_path.exists():
 subdirs = list(dicom_path.iterdir())[:10]
 print(f" First 10 items in {dicom_path}:")
 for d in subdirs:
 print(f" {d.name}")
 else:
 print(f" DICOM path does not exist: {dicom_path}")
 
 raise ValueError("No valid ROI samples found! Check DICOM path configuration.")
 
 # Convert to DataFrame
 samples_df = pd.DataFrame(valid_samples)
 
 # Class distribution
 print(f"\n Class Distribution:")
 print(f" Benign (0): {(samples_df['label'] == 0).sum()}")
 print(f" Malignant (1): {(samples_df['label'] == 1).sum()}")
 
 # Lesion type distribution
 print(f"\n Lesion Type Distribution:")
 for lt in samples_df['lesion_type'].unique():
 count = (samples_df['lesion_type'] == lt).sum()
 print(f" {lt.capitalize()}: {count}")
 
 return samples_df

# Load data
samples_df = load_cbis_ddsm_data(config)
print("="*70)

In [ ]:
# Create Data Generators (V3 Compatible)
# Uses case_dir-based image loading like V3

from tensorflow.keras.utils import Sequence
import cv2

class CBISDDSMGenerator(Sequence):
 """
 Custom data generator for CBIS-DDSM DICOM images.
 
 Uses case_dir-based loading (V3 approach) for proper ROI crops.
 """
 
 def __init__(self, df, config, batch_size=32, img_size=(224, 224), 
 augment=False, shuffle=True):
 self.df = df.copy()
 self.config = config
 self.batch_size = batch_size
 self.img_size = img_size
 self.augment = augment
 self.shuffle = shuffle
 self.indices = np.arange(len(self.df))
 
 if self.shuffle:
 np.random.shuffle(self.indices)
 
 def __len__(self):
 return int(np.ceil(len(self.df) / self.batch_size))
 
 def __getitem__(self, idx):
 batch_indices = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
 batch_df = self.df.iloc[batch_indices]
 
 X = []
 y = []
 
 for _, row in batch_df.iterrows():
 # Use V3-compatible load_dicom_image with case_dir
 case_dir = row['case_dir']
 img = load_dicom_image(case_dir, self.config.DICOM_PATH, self.img_size)
 
 if img is not None:
 # Augmentation (training only)
 if self.augment:
 img = self._augment(img)
 
 X.append(img)
 y.append(row['label'])
 
 if len(X) == 0:
 # Return empty batch with correct shape
 return np.zeros((0, *self.img_size, 3)), np.zeros((0,))
 
 return np.array(X), np.array(y)
 
 def _augment(self, img):
 """
 Apply conservative data augmentation for medical imaging.
 
 Based on literature: limited rotation, horizontal flip only.
 """
 # Random horizontal flip (mammograms can be flipped)
 if np.random.random() > 0.5:
 img = np.fliplr(img).copy()
 
 # Random rotation (±10 degrees - conservative for medical)
 angle = np.random.uniform(-10, 10)
 h, w = img.shape[:2]
 M = cv2.getRotationMatrix2D((w/2, h/2), angle, 1.0)
 img = cv2.warpAffine(img, M, (w, h))
 
 # Slight brightness adjustment (±5%)
 brightness = np.random.uniform(0.95, 1.05)
 img = img * brightness
 
 return img
 
 def on_epoch_end(self):
 if self.shuffle:
 np.random.shuffle(self.indices)

print(" Custom data generator defined (V3 compatible)")

In [ ]:
# Split Data and Create Generators

from sklearn.model_selection import train_test_split

# Stratified split to maintain class balance
train_df, temp_df = train_test_split(
 samples_df, 
 test_size=(config.VAL_SPLIT + config.TEST_SPLIT),
 stratify=samples_df['label'],
 random_state=42
)

val_df, test_df = train_test_split(
 temp_df,
 test_size=config.TEST_SPLIT / (config.VAL_SPLIT + config.TEST_SPLIT),
 stratify=temp_df['label'],
 random_state=42
)

print(f"\n Data Split (Stratified):")
print(f" Training: {len(train_df)} samples")
print(f" Validation: {len(val_df)} samples")
print(f" Test: {len(test_df)} samples")

# Verify class distribution
for name, df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
 n_benign = (df['label'] == 0).sum()
 n_malignant = (df['label'] == 1).sum()
 print(f" {name}: Benign={n_benign}, Malignant={n_malignant}")

# Create generators (now with config for V3-compatible loading)
train_generator = CBISDDSMGenerator(
 train_df,
 config=config, # Pass config for DICOM path
 batch_size=config.BATCH_SIZE,
 img_size=config.IMG_SIZE,
 augment=True,
 shuffle=True
)

val_generator = CBISDDSMGenerator(
 val_df,
 config=config,
 batch_size=config.BATCH_SIZE,
 img_size=config.IMG_SIZE,
 augment=False,
 shuffle=False
)

test_generator = CBISDDSMGenerator(
 test_df,
 config=config,
 batch_size=config.BATCH_SIZE,
 img_size=config.IMG_SIZE,
 augment=False,
 shuffle=False
)

print(f"\n Data generators created")
print(f" Train batches: {len(train_generator)}")
print(f" Val batches: {len(val_generator)}")
print(f" Test batches: {len(test_generator)}")

In [ ]:
# Compute Class Weights (for reference) & Focal Loss Alpha

from sklearn.utils.class_weight import compute_class_weight

# Compute balanced class weights
balanced_weights = compute_class_weight(
 class_weight='balanced',
 classes=np.array([0, 1]),
 y=train_df['label'].values
)

# V4: Store for REFERENCE ONLY - NOT used in model.fit()!
config.CLASS_WEIGHTS_REFERENCE = {
 0: balanced_weights[0],
 1: balanced_weights[1] * config.MALIGNANT_WEIGHT_MULTIPLIER
}

print("\n" + "="*70)
print(" V4 CLASS WEIGHTING STRATEGY")
print("="*70)
print(f"\n Training Class Distribution:")
print(f" Benign (0): {(train_df['label'] == 0).sum()} ({(train_df['label'] == 0).mean()*100:.1f}%)")
print(f" Malignant (1): {(train_df['label'] == 1).sum()} ({(train_df['label'] == 1).mean()*100:.1f}%)")

print(f"\n V4 FIX: Using ONLY Focal Loss for class weighting")
print(f" Focal Loss α: {config.FOCAL_LOSS_ALPHA} (75% weight on malignant)")
print(f" Focal Loss γ: {config.FOCAL_LOSS_GAMMA} (focus on hard examples)")
print(f" class_weight in model.fit(): DISABLED (was {config.CLASS_WEIGHTS_REFERENCE})")

print(f"\n Why? V3 used BOTH → ~10x effective weight → catastrophic forgetting")
print("="*70)

In [ ]:
# Define Focal Loss (Literature-Backed)

class FocalLoss(keras.losses.Loss):
 """Focal Loss for imbalanced classification
 
 Formula: FL(pt) = -α * (1-pt)^γ * log(pt)
 
 - α: Class balance factor (0.75 = 75% weight on positive/malignant)
 - γ: Focus factor (2.0 = focus on hard examples)
 
 This REPLACES class_weight in model.fit() to avoid double weighting!
 """
 
 def __init__(self, gamma=2.0, alpha=0.5, name="focal_loss"):
 super().__init__(name=name)
 self.gamma = gamma
 self.alpha = alpha
 
 def call(self, y_true, y_pred):
 # Clip predictions to avoid log(0)
 epsilon = tf.keras.backend.epsilon()
 y_pred = tf.clip_by_value(y_pred, epsilon, 1 - epsilon)
 
 # Calculate focal loss
 y_true = tf.cast(y_true, tf.float32)
 
 # Cross entropy
 ce = -y_true * tf.math.log(y_pred) - (1 - y_true) * tf.math.log(1 - y_pred)
 
 # Focal term: (1 - pt)^gamma
 pt = y_true * y_pred + (1 - y_true) * (1 - y_pred)
 focal_weight = tf.pow(1 - pt, self.gamma)
 
 # Alpha balancing
 alpha_weight = y_true * self.alpha + (1 - y_true) * (1 - self.alpha)
 
 # Final loss
 focal_loss = alpha_weight * focal_weight * ce
 
 return tf.reduce_mean(focal_loss)

def focal_loss(gamma=2.0, alpha=0.5):
 """Focal loss as a function for model.compile()"""
 def loss_fn(y_true, y_pred):
 epsilon = tf.keras.backend.epsilon()
 y_pred = tf.clip_by_value(y_pred, epsilon, 1 - epsilon)
 y_true = tf.cast(y_true, tf.float32)
 
 ce = -y_true * tf.math.log(y_pred) - (1 - y_true) * tf.math.log(1 - y_pred)
 pt = y_true * y_pred + (1 - y_true) * (1 - y_pred)
 focal_weight = tf.pow(1 - pt, gamma)
 alpha_weight = y_true * alpha + (1 - y_true) * (1 - alpha)
 
 return tf.reduce_mean(alpha_weight * focal_weight * ce)
 
 return loss_fn

print(" Focal Loss defined")
print(f" α = {config.FOCAL_LOSS_ALPHA} (malignant weight)")
print(f" γ = {config.FOCAL_LOSS_GAMMA} (focus factor)")

In [ ]:
# Build Model (Literature-Backed Architecture)
# Per ILB-BCD paper (99.16% accuracy):
# - DenseNet121 base with ImageNet weights
# - Custom head: GAP → BN → Dense(2048, L1/L2=0.01) → BN → Dense(1)

def build_model(config):
 """Build V4 model with literature-backed architecture"""
 
 print("\n" + "="*70)
 print(" BUILDING V4 MODEL (Literature-Backed)")
 print("="*70)
 
 # Load DenseNet121 base with ImageNet weights
 base_model = DenseNet121(
 include_top=False,
 weights='imagenet',
 input_shape=(config.IMG_SIZE[0], config.IMG_SIZE[1], 3),
 pooling=None
 )
 
 print(f"\n DenseNet121 base loaded")
 print(f" Total layers: {len(base_model.layers)}")
 
 # Freeze base initially
 base_model.trainable = False
 
 # Build custom classification head (per ILB-BCD paper)
 x = base_model.output
 x = layers.GlobalAveragePooling2D(name='gap')(x)
 x = layers.BatchNormalization(name='bn_1')(x)
 
 # Dense layer with L1/L2 regularization (per ILB-BCD: 0.01)
 x = layers.Dense(
 2048,
 activation='relu',
 kernel_regularizer=regularizers.l1_l2(
 l1=config.L1_REGULARIZATION,
 l2=config.L2_REGULARIZATION
 ),
 name='dense_2048'
 )(x)
 
 x = layers.BatchNormalization(name='bn_2')(x)
 x = layers.Dropout(config.DROPOUT_RATE, name='dropout')(x)
 
 # Output layer
 output = layers.Dense(
 1,
 activation='sigmoid',
 name='output'
 )(x)
 
 model = models.Model(inputs=base_model.input, outputs=output, name='DenseNet121_V4')
 
 # Print model summary
 trainable = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
 total = model.count_params()
 
 print(f"\n Model Parameters:")
 print(f" Total: {total:,}")
 print(f" Trainable: {trainable:,} ({trainable/total*100:.1f}%)")
 print(f" Non-trainable: {total - trainable:,}")
 
 print(f"\n Architecture (per ILB-BCD paper):")
 print(f" Base: DenseNet121 (ImageNet)")
 print(f" Head: GAP → BN → Dense(2048, L1/L2={config.L1_REGULARIZATION}) → BN → Dropout({config.DROPOUT_RATE}) → Dense(1, sigmoid)")
 print("="*70)
 
 return model, base_model

# Build model
model, base_model = build_model(config)

In [ ]:
# Define Callbacks

def get_callbacks(stage_name, config, min_epochs=20):
 """Get V4 callbacks with improved settings"""
 
 callbacks = [
 # Save best model based on val_auc (most important for medical imaging)
 ModelCheckpoint(
 str(config.CHECKPOINT_PATH / f'{stage_name}_best_model.h5'),
 monitor='val_auc',
 mode='max',
 save_best_only=True,
 verbose=1
 ),
 # Also save based on val_loss for comparison
 ModelCheckpoint(
 str(config.CHECKPOINT_PATH / f'{stage_name}_best_loss.h5'),
 monitor='val_loss',
 mode='min',
 save_best_only=True,
 verbose=0
 ),
 # Reduce LR on plateau
 ReduceLROnPlateau(
 monitor='val_loss',
 factor=config.LR_REDUCE_FACTOR,
 patience=config.LR_REDUCE_PATIENCE,
 min_lr=1e-8,
 verbose=1
 ),
 # Early stopping (with min_epochs)
 EarlyStopping(
 monitor='val_auc',
 mode='max',
 patience=config.EARLY_STOP_PATIENCE,
 restore_best_weights=True,
 verbose=1,
 start_from_epoch=min_epochs # Don't stop before min_epochs
 )
 ]
 
 return callbacks

print(" V4 Callbacks defined")

In [ ]:
# Diagnostic Callback

class DiagnosticCallback(keras.callbacks.Callback):
 """Real-time training diagnostics for medical imaging"""
 
 def __init__(self, val_generator, check_interval=5):
 super().__init__()
 self.val_generator = val_generator
 self.check_interval = check_interval
 self.history = {'epoch': [], 'pred_mean': [], 'pred_std': [], 
 'pct_malignant_pred': [], 'sensitivity': [], 'specificity': []}
 
 def on_epoch_end(self, epoch, logs=None):
 if (epoch + 1) % self.check_interval == 0:
 print(f"\n{'='*60}")
 print(f" DIAGNOSTIC CHECK - Epoch {epoch + 1}")
 print(f"{'='*60}")
 
 # Get predictions on validation set
 y_true = []
 y_pred = []
 
 for i in range(len(self.val_generator)):
 X_batch, y_batch = self.val_generator[i]
 preds = self.model.predict(X_batch, verbose=0)
 y_true.extend(y_batch.flatten())
 y_pred.extend(preds.flatten())
 
 y_true = np.array(y_true)
 y_pred = np.array(y_pred)
 
 # Prediction statistics
 pred_mean = y_pred.mean()
 pred_std = y_pred.std()
 pct_pred_malignant = (y_pred > 0.5).mean() * 100
 
 # Classification metrics at threshold 0.5
 y_pred_binary = (y_pred > 0.5).astype(int)
 
 tp = ((y_pred_binary == 1) & (y_true == 1)).sum()
 tn = ((y_pred_binary == 0) & (y_true == 0)).sum()
 fp = ((y_pred_binary == 1) & (y_true == 0)).sum()
 fn = ((y_pred_binary == 0) & (y_true == 1)).sum()
 
 sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
 specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
 
 # Store history
 self.history['epoch'].append(epoch + 1)
 self.history['pred_mean'].append(pred_mean)
 self.history['pred_std'].append(pred_std)
 self.history['pct_malignant_pred'].append(pct_pred_malignant)
 self.history['sensitivity'].append(sensitivity)
 self.history['specificity'].append(specificity)
 
 # Print diagnostics
 print(f"\n Prediction Distribution:")
 print(f" Mean prediction: {pred_mean:.4f} (ideal: ~0.4-0.6)")
 print(f" Std deviation: {pred_std:.4f} (ideal: >0.2)")
 print(f" % predicted malignant: {pct_pred_malignant:.1f}%")
 
 print(f"\n Classification Metrics (threshold=0.5):")
 print(f" Sensitivity: {sensitivity*100:.1f}% (target: >96%)")
 print(f" Specificity: {specificity*100:.1f}% (target: >82%)")
 print(f" TP={tp}, TN={tn}, FP={fp}, FN={fn}")
 
 # Warnings
 if pred_std < 0.1:
 print(f"\n WARNING: Low prediction variance ({pred_std:.3f})")
 if sensitivity < 0.5:
 print(f"\n CRITICAL: Sensitivity too low ({sensitivity*100:.1f}%)")
 
 print(f"{'='*60}\n")

# Create diagnostic callback
diagnostic_cb = DiagnosticCallback(val_generator, check_interval=5)
print(" Diagnostic callback created")

## Stage 1: Train Classification Head Only

In [ ]:
# STAGE 1 - Train Head Only
# Per ILB-BCD paper: Train head first with higher LR

print("\n" + "="*70)
print(" STAGE 1: TRAIN CLASSIFICATION HEAD (Frozen Base)")
print("="*70)

# Ensure base is frozen
base_model.trainable = False

# V4: Compile with gradient clipping
optimizer = Adam(
 learning_rate=config.STAGE1_LR,
 clipnorm=config.GRADIENT_CLIPNORM # V4: Gradient clipping!
)

model.compile(
 optimizer=optimizer,
 loss=focal_loss(gamma=config.FOCAL_LOSS_GAMMA, alpha=config.FOCAL_LOSS_ALPHA),
 metrics=[
 BinaryAccuracy(name='accuracy'),
 Precision(name='precision'),
 Recall(name='recall'),
 AUC(name='auc')
 ]
)

print(f"\n Configuration:")
print(f" Epochs: {config.STAGE1_EPOCHS}")
print(f" Learning Rate: {config.STAGE1_LR}")
print(f" Gradient Clipping: clipnorm={config.GRADIENT_CLIPNORM}")
print(f" Loss: Focal Loss (α={config.FOCAL_LOSS_ALPHA}, γ={config.FOCAL_LOSS_GAMMA})")
print(f" class_weight: DISABLED (using focal loss α instead)")
print("="*70 + "\n")

# Train Stage 1
history_stage1 = model.fit(
 train_generator,
 validation_data=val_generator,
 epochs=config.STAGE1_EPOCHS,
 # V4 FIX: NO class_weight! Using focal loss alpha instead
 callbacks=get_callbacks('stage1', config, min_epochs=config.MIN_EPOCHS_STAGE1) + [diagnostic_cb],
 verbose=1
)

print("\n" + "="*70)
print(" STAGE 1 COMPLETE")
print(f" Best val_auc: {max(history_stage1.history['val_auc']):.4f}")
print("="*70)

## Stage 2a: Gradual Unfreezing (Last Dense Block)

In [ ]:
# STAGE 2a - Unfreeze Last Dense Block

print("\n" + "="*70)
print(" STAGE 2a: GRADUAL UNFREEZING (Last Dense Block)")
print("="*70)

# Unfreeze last dense block (conv5_block*)
base_model.trainable = True

# Freeze all layers first
for layer in base_model.layers:
 layer.trainable = False

# Unfreeze from conv5_block1 onwards (last dense block)
unfreeze_from = 'conv5_block1'
found = False
unfrozen = 0

for layer in base_model.layers:
 if unfreeze_from in layer.name or found:
 layer.trainable = True
 found = True
 unfrozen += 1

print(f"\n Layer Status:")
print(f" Frozen: {len(base_model.layers) - unfrozen}")
print(f" Unfrozen: {unfrozen} (last dense block + BN)")

# Recompile with lower LR and gradient clipping
optimizer = Adam(
 learning_rate=config.STAGE2A_LR,
 clipnorm=config.GRADIENT_CLIPNORM
)

model.compile(
 optimizer=optimizer,
 loss=focal_loss(gamma=config.FOCAL_LOSS_GAMMA, alpha=config.FOCAL_LOSS_ALPHA),
 metrics=[
 BinaryAccuracy(name='accuracy'),
 Precision(name='precision'),
 Recall(name='recall'),
 AUC(name='auc')
 ]
)

print(f"\n Configuration:")
print(f" Epochs: {config.STAGE2A_EPOCHS}")
print(f" Learning Rate: {config.STAGE2A_LR}")
print(f" class_weight: DISABLED")
print("="*70 + "\n")

# Train Stage 2a
history_stage2a = model.fit(
 train_generator,
 validation_data=val_generator,
 epochs=config.STAGE2A_EPOCHS,
 # V4 FIX: NO class_weight!
 callbacks=get_callbacks('stage2a', config, min_epochs=config.MIN_EPOCHS_STAGE2A) + [diagnostic_cb],
 verbose=1
)

print("\n" + "="*70)
print(" STAGE 2a COMPLETE")
print(f" Best val_auc: {max(history_stage2a.history['val_auc']):.4f}")
print("="*70)

## Stage 2b: Partial Fine-Tuning (V4 FIX: Keep Early Layers Frozen)

In [ ]:
# STAGE 2b - V4 FIX: PARTIAL FINE-TUNING
# V4 KEY FIX: Keep first 200 layers FROZEN to prevent catastrophic forgetting!
# These contain generic features (edges, textures) that transfer well.

print("\n" + "="*70)
print(" STAGE 2b: PARTIAL FINE-TUNING (V4 FIX)")
print("="*70)

# V4 FIX: Unfreeze from layer 200 onwards (NOT all layers!)
base_model.trainable = True

frozen_count = 0
for i, layer in enumerate(base_model.layers):
 if i < config.FREEZE_FIRST_N_LAYERS:
 layer.trainable = False
 frozen_count += 1
 else:
 layer.trainable = True

# Count trainable params
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
total_params = model.count_params()

print(f"\n V4 FIX: Partial Unfreezing")
print(f" Frozen layers: {frozen_count} (first {config.FREEZE_FIRST_N_LAYERS})")
print(f" Unfrozen layers: {len(base_model.layers) - frozen_count}")
print(f" Trainable params: {trainable_params:,} ({trainable_params/total_params*100:.1f}%)")

# V4 FIX: Use VERY low learning rate
optimizer = Adam(
 learning_rate=config.STAGE2B_LR, # V4: 1e-6 (was 1e-5 in V3!)
 clipnorm=config.GRADIENT_CLIPNORM
)

model.compile(
 optimizer=optimizer,
 loss=focal_loss(gamma=config.FOCAL_LOSS_GAMMA, alpha=config.FOCAL_LOSS_ALPHA),
 metrics=[
 BinaryAccuracy(name='accuracy'),
 Precision(name='precision'),
 Recall(name='recall'),
 AUC(name='auc')
 ]
)

print(f"\n V4 Configuration:")
print(f" Epochs: {config.STAGE2B_EPOCHS}")
print(f" Learning Rate: {config.STAGE2B_LR} (10x lower than V3!)")
print(f" Frozen layers: first {config.FREEZE_FIRST_N_LAYERS}")
print(f" Gradient clipping: clipnorm={config.GRADIENT_CLIPNORM}")
print(f" class_weight: DISABLED (V4 fix)")
print("="*70 + "\n")

# Train Stage 2b
history_stage2b = model.fit(
 train_generator,
 validation_data=val_generator,
 epochs=config.STAGE2B_EPOCHS,
 # V4 FIX: NO class_weight!
 callbacks=get_callbacks('stage2b', config, min_epochs=config.MIN_EPOCHS_STAGE2B) + [diagnostic_cb],
 verbose=1
)

print("\n" + "="*70)
print(" STAGE 2b COMPLETE")
print(f" Best val_auc: {max(history_stage2b.history['val_auc']):.4f}")
print("="*70)

## Evaluation & Threshold Optimization

In [ ]:
# Final Evaluation

print("\n" + "="*70)
print(" FINAL EVALUATION")
print("="*70)

# Load best model
best_model_path = config.CHECKPOINT_PATH / 'stage2b_best_model.h5'
if best_model_path.exists():
 model.load_weights(str(best_model_path))
 print(f" Loaded best model from {best_model_path}")

# Get predictions on test set
y_true = []
y_pred = []

for i in range(len(test_generator)):
 X_batch, y_batch = test_generator[i]
 preds = model.predict(X_batch, verbose=0)
 y_true.extend(y_batch.flatten())
 y_pred.extend(preds.flatten())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate AUC
auc = roc_auc_score(y_true, y_pred)
print(f"\n Test Set AUC-ROC: {auc:.4f}")

# Find optimal threshold
fpr, tpr, thresholds = roc_curve(y_true, y_pred)

# Youden's J statistic
j_scores = tpr - fpr
optimal_idx = np.argmax(j_scores)
optimal_threshold = thresholds[optimal_idx]

print(f"\n Optimal Threshold (Youden's J): {optimal_threshold:.3f}")

# Classification at optimal threshold
y_pred_opt = (y_pred >= optimal_threshold).astype(int)

tp = ((y_pred_opt == 1) & (y_true == 1)).sum()
tn = ((y_pred_opt == 0) & (y_true == 0)).sum()
fp = ((y_pred_opt == 1) & (y_true == 0)).sum()
fn = ((y_pred_opt == 0) & (y_true == 1)).sum()

sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
npv = tn / (tn + fn) if (tn + fn) > 0 else 0
fnr = fn / (fn + tp) if (fn + tp) > 0 else 0

print(f"\n Test Set Metrics @ threshold={optimal_threshold:.3f}:")
print(f" Sensitivity: {sensitivity*100:.1f}% (target: 96.2%)")
print(f" Specificity: {specificity*100:.1f}% (target: 82.5%)")
print(f" NPV: {npv*100:.1f}% (target: 99.1%)")
print(f" FNR: {fnr*100:.1f}% (target: 3.8%)")
print(f" AUC-ROC: {auc:.4f} (target: 0.93)")

print("\n" + "="*70)

In [ ]:
# Plot ROC Curve & Training History

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
ax1 = axes[0]
ax1.plot(fpr, tpr, 'b-', label=f'V4 Model (AUC = {auc:.3f})')
ax1.plot([0, 1], [0, 1], 'k--', label='Random')
ax1.scatter([1-specificity], [sensitivity], c='red', s=100, marker='*', 
 label=f'Optimal @ {optimal_threshold:.3f}')
ax1.set_xlabel('False Positive Rate (1 - Specificity)')
ax1.set_ylabel('True Positive Rate (Sensitivity)')
ax1.set_title('ROC Curve - V4 Literature-Backed Model')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Training History (AUC)
ax2 = axes[1]

# Combine all histories
all_val_auc = []
all_val_auc.extend(history_stage1.history['val_auc'])
all_val_auc.extend(history_stage2a.history['val_auc'])
all_val_auc.extend(history_stage2b.history['val_auc'])

epochs = range(1, len(all_val_auc) + 1)
ax2.plot(epochs, all_val_auc, 'b-', label='Val AUC')

# Mark stage boundaries
stage1_end = config.STAGE1_EPOCHS
stage2a_end = stage1_end + config.STAGE2A_EPOCHS
ax2.axvline(x=stage1_end, color='g', linestyle='--', alpha=0.5, label='Stage 2a Start')
ax2.axvline(x=stage2a_end, color='r', linestyle='--', alpha=0.5, label='Stage 2b Start')

ax2.set_xlabel('Epoch')
ax2.set_ylabel('Validation AUC')
ax2.set_title('Training History - V4 Model')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(config.RESULTS_PATH / 'v4_training_results.png'), dpi=150)
plt.show()

print(f"\n Results saved to {config.RESULTS_PATH}")

In [ ]:
# Save Final Results

import json

# Save results
results = {
 'version': 'V4_LITERATURE_BACKED',
 'test_auc': float(auc),
 'optimal_threshold': float(optimal_threshold),
 'sensitivity': float(sensitivity),
 'specificity': float(specificity),
 'npv': float(npv),
 'fnr': float(fnr),
 'configuration': {
 'stage1_lr': config.STAGE1_LR,
 'stage2a_lr': config.STAGE2A_LR,
 'stage2b_lr': config.STAGE2B_LR,
 'focal_loss_alpha': config.FOCAL_LOSS_ALPHA,
 'focal_loss_gamma': config.FOCAL_LOSS_GAMMA,
 'freeze_first_n_layers': config.FREEZE_FIRST_N_LAYERS,
 'l1_regularization': config.L1_REGULARIZATION,
 'l2_regularization': config.L2_REGULARIZATION,
 'gradient_clipnorm': config.GRADIENT_CLIPNORM,
 'class_weight_in_fit': config.USE_CLASS_WEIGHT_IN_FIT
 },
 'v4_fixes': [
 'Removed class_weight from model.fit() - using only focal loss alpha',
 'Reduced Stage 2b LR from 1e-5 to 1e-6',
 f'Keep first {config.FREEZE_FIRST_N_LAYERS} layers frozen in Stage 2b',
 'Increased L1/L2 regularization to 0.01 (per ILB-BCD paper)',
 'Added gradient clipping (clipnorm=1.0)'
 ]
}

with open(config.RESULTS_PATH / 'v4_final_results.json', 'w') as f:
 json.dump(results, f, indent=2)

print("\n" + "="*70)
print(" V4 LITERATURE-BACKED RESULTS SUMMARY")
print("="*70)
print(f"\n Target vs Achieved:")
print(f" Sensitivity: {sensitivity*100:.1f}% (target: 96.2%)")
print(f" Specificity: {specificity*100:.1f}% (target: 82.5%)")
print(f" NPV: {npv*100:.1f}% (target: 99.1%)")
print(f" FNR: {fnr*100:.1f}% (target: 3.8%)")
print(f" AUC-ROC: {auc:.4f} (target: 0.93)")
print(f"\n V4 Key Fixes Applied:")
for fix in results['v4_fixes']:
 print(f" {fix}")
print(f"\n Results saved to: {config.RESULTS_PATH}")
print("="*70)

## V4 Changes Summary

### Literature References
1. **McKinney et al., Nature 2020** - Google Health mammography AI
2. **Swapna Rani et al., JATIT 2025** - ILB-BCD algorithm (99.16% accuracy)
3. **Wu et al., IEEE TMI 2019** - Two-stage training

### V4 Key Fixes (from V3 analysis)

| Issue | V3 Problem | V4 Fix |
|-------|------------|--------|
| **Double Weighting** | class_weight + focal_loss α | ONLY focal loss (α=0.75) |
| **Stage 2b LR** | 1e-5 (catastrophic forgetting) | 1e-6 (10x lower) |
| **Unfreezing** | All 9M params at once | Keep first 200 layers frozen |
| **Regularization** | L1=1e-5, L2=1e-4 | L1=L2=0.01 (per literature) |
| **Gradient Clipping** | None | clipnorm=1.0 |

### Why These Fixes?

1. **Double Weighting** caused ~10x effective malignant weight:
 - class_weight: 3.6x
 - focal_loss α: 0.75 → 3x
 - Combined: ~10x → extreme malignant bias → specificity collapse

2. **Stage 2b LR=1e-5** was too high for unfreezing 9M parameters:
 - Caused val_loss to explode (0.12 → 0.75)
 - val_auc dropped from 0.73 → 0.48

3. **Early layers** contain generic features (edges, textures):
 - These transfer well and should stay frozen
 - Only later layers need domain adaptation

4. **L1/L2=0.01** per ILB-BCD paper:
 - Prevents overfitting
 - Helps generalization